# TIME TRAVEL

In [0]:
# ============================================================
# 09_TIME_TRAVEL
#
# Purpose:
# Learn Delta Lake features
#
# Concepts:
#
# - Delta Transaction Log
# - DESCRIBE HISTORY
# - Time Travel
# - RESTORE TABLE
# - VACUUM
# - OPTIMIZE
# - ZORDER
#
# ============================================================



from delta.tables import DeltaTable
from pyspark.sql.functions import *





# ============================================================
# 1. Configuration
# ============================================================


table_name = "ecommerce.gold.customer_summary"





# ============================================================
# 2. Check Delta Table Information
#
# Delta tables store:
#
# Data files
# +
# Transaction logs
#
# Location:
#
# _delta_log
#
# ============================================================


print("Current Table Data")


display(
    spark.table(table_name)
    .limit(10)
)





# ============================================================
# 3. DESCRIBE HISTORY
#
# Shows every operation performed
#
# Examples:
#
# WRITE
# UPDATE
# DELETE
# MERGE
# OPTIMIZE
#
# ============================================================


print("Delta History")


spark.sql(
f"""
DESCRIBE HISTORY {table_name}
"""
).show(
    truncate=False
)





# ============================================================
# 4. Store Current Version
#
# Every Delta operation creates
# a new version number
#
# ============================================================


history = spark.sql(
f"""
DESCRIBE HISTORY {table_name}
"""
)



current_version = (

    history

    .select(
        max("version")
    )

    .collect()[0][0]

)



print(
    "Current Version:",
    current_version
)





# ============================================================
# 5. Create a Change
#
# We modify the table so
# we can travel back
#
# ============================================================



df = spark.table(
    table_name
)



changed_df = (

    df

    .withColumn(
        "test_column",
        lit("new_change")
    )

)



changed_df.write \
.format("delta") \
.mode("overwrite") \
.option(
    "overwriteSchema",
    "true"
) \
.saveAsTable(
    table_name
)



print(
    "New version created"
)





# ============================================================
# 6. Check New History
# ============================================================


spark.sql(
f"""
DESCRIBE HISTORY {table_name}
"""
).show(
    truncate=False
)





# ============================================================
# 7. TIME TRAVEL USING VERSION
#
# Read old table version
#
# ============================================================



old_version_df = spark.read \
.format("delta") \
.option(
    "versionAsOf",
    current_version
) \
.table(
    table_name
)



print(
    "Old Version Data"
)



display(
    old_version_df.limit(10)
)





# ============================================================
# 8. TIME TRAVEL USING TIMESTAMP
#
# Read table at a specific time
#
# ============================================================



timestamp_df = spark.read \
.format("delta") \
.option(
    "timestampAsOf",
    "2026-07-15"
) \
.table(
    table_name
)



display(
    timestamp_df.limit(10)
)





# ============================================================
# 9. RESTORE TABLE
#
# Restore table back to old version
#
# Useful when:
#
# Bad data was loaded
# Wrong transformation happened
#
# ============================================================



spark.sql(
f"""
RESTORE TABLE {table_name}
TO VERSION AS OF {current_version}
"""
)



print(
    "Table Restored"
)





# ============================================================
# 10. DELETE DATA FOR VACUUM DEMO
#
# Create temporary changes
#
# ============================================================


spark.sql(
f"""
DELETE FROM {table_name}
WHERE customer_id IS NULL
"""
)



print(
    "Delete operation completed"
)





# ============================================================
# 11. DESCRIBE HISTORY AGAIN
# ============================================================


spark.sql(
f"""
DESCRIBE HISTORY {table_name}
"""
).show(
    truncate=False
)





# ============================================================
# 12. OPTIMIZE TABLE
#
# Problem:
#
# Many small files decrease performance
#
# OPTIMIZE combines files
#
# ============================================================



spark.sql(
f"""
OPTIMIZE {table_name}
"""
)



print(
    "Optimize completed"
)





# ============================================================
# 13. ZORDER
#
# Improves data skipping
#
# Similar to indexing concept
#
# ============================================================



spark.sql(
f"""
OPTIMIZE {table_name}
ZORDER BY(customer_id)
"""
)



print(
    "ZORDER completed"
)





# ============================================================
# 14. VACUUM
#
# Removes old unused files
#
# Default retention:
# 7 days
#
# ============================================================



spark.sql(
f"""
VACUUM {table_name}
"""
)



print(
    "Vacuum completed"
)





# ============================================================
# 15. Final History Check
# ============================================================


spark.sql(
f"""
DESCRIBE HISTORY {table_name}
"""
).show(
    truncate=False
)

Current Table Data


customer_id,first_name,last_name,country,membership,total_orders,total_spent,average_order_value
186,Kyle,Martinez,PAKISTAN,Bronze,96,330005.81,3437.560521
343,Aaron,Garrison,PAKISTAN,Silver,97,290788.78,2997.822474
782,Ashley,Perry,PAKISTAN,Bronze,89,276685.96,3108.831011
784,Eric,Gibson,PAKISTAN,Bronze,100,252928.09,2529.280900
225,William,Torres,PAKISTAN,Silver,108,286957.71,2657.015833
660,Stephen,Smith,PAKISTAN,Silver,92,297075.45,3229.080978
633,John,Medina,PAKISTAN,Bronze,98,272841.76,2784.099592
826,Brenda,Clark,PAKISTAN,Silver,98,281109.93,2868.468673
641,Robert,Chavez,PAKISTAN,Silver,91,283807.31,3118.761648
881,Amanda,Brown,PAKISTAN,Gold,107,321458.87,3004.288505


Delta History
+-------+-------------------+--------------+-----------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+-----------------+------------------------------------+------------------------+-----------+-----------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------------+
|version|timestamp          |userId        |userName               |operation                        |operationParameters                                                                                                           

customer_id,first_name,last_name,country,membership,total_orders,total_spent,average_order_value
186,Kyle,Martinez,PAKISTAN,Bronze,96,330005.81,3437.560521
343,Aaron,Garrison,PAKISTAN,Silver,97,290788.78,2997.822474
782,Ashley,Perry,PAKISTAN,Bronze,89,276685.96,3108.831011
784,Eric,Gibson,PAKISTAN,Bronze,100,252928.09,2529.280900
225,William,Torres,PAKISTAN,Silver,108,286957.71,2657.015833
660,Stephen,Smith,PAKISTAN,Silver,92,297075.45,3229.080978
633,John,Medina,PAKISTAN,Bronze,98,272841.76,2784.099592
826,Brenda,Clark,PAKISTAN,Silver,98,281109.93,2868.468673
641,Robert,Chavez,PAKISTAN,Silver,91,283807.31,3118.761648
881,Amanda,Brown,PAKISTAN,Gold,107,321458.87,3004.288505


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5913672782069867>, line 261
    240 # ============================================================
    241 # 8. TIME TRAVEL USING TIMESTAMP
    242 #
    243 # Read table at a specific time
    244 #
    245 # ============================================================
    249 timestamp_df = spark.read \
    250 .format("delta") \
    251 .option(
   (...)
    256     table_name
    257 )
--> 261 display(
    262     timestamp_df.limit(10)
    263 )
    269 # ============================================================
    270 # 9. RESTORE TABLE
    271 #
   (...)
    278 #
    279 # ============================================================
    283 spark.sql(
    284 f"""
    285 RESTORE TABLE {table_name}
    286 TO VERSION AS OF {current_version}
    287 """
    288 )

File /databricks/python_shell/lib/dbruntime/disp

In [0]:
%sql
DESCRIBE HISTORY ecommerce.gold.customer_summary